In [2]:
from elasticsearch import Elasticsearch
from deepface import DeepFace
import matplotlib.pyplot as plt
from retinaface import RetinaFace
import os
import time
import glob
from faker import Faker
from uuid import uuid4
from datetime import datetime
import cv2 as cv

In [5]:
es = Elasticsearch(hosts=['http://localhost:9200'], http_auth=('elastic', 'DkIed99SCb'))

In [6]:
model_name = 'Facenet512'
target_size = (160, 160)
embedding_size = 512

In [25]:
mapping = {
    "mappings": {
        "properties": {
            "id": {"type": "keyword"},
            "title_vector": {
                "type": "dense_vector",
                "dims": embedding_size
            },
            "title_name": {"type": "keyword"},
            "description": {"type": "keyword"},
            "img_path": {"type": "keyword"},
            "gender": {"type": "keyword"},
            "dob": {"type": "date"},
            "birth_place": {"type": "keyword"},
            "image_url": {"type": "keyword"},
            "face_path": {"type": "keyword"},
            "identified_age": {"type": "keyword"},
            "identified_gender": {"type": "keyword"},
            "identified_race": {"type": "keyword"},
            "identified_emotion": {"type": "keyword"},
            "username": {"type": "keyword"},
            "created_at": {"type": "date"}
        }
    }
}
   
es.indices.create(index="suspects", body=mapping)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'suspects'})

In [10]:
dir_path = "public/police/*.*"
res = glob.glob(dir_path, recursive=True)
files = []
for item in res:
    if item.endswith('.jpg'):
        files.append(item)
print('total files:', len(files))

total files: 5910


In [ ]:
tic = time.time()
fake = Faker()

for img_path in files:
    print('indexing', img_path, 'index', files.index(img_path)+1, 'of', len(files))
    embedding_objs = DeepFace.represent(img_path=img_path, model_name=model_name, detector_backend='retinaface', align=True)
    embedding = embedding_objs[0]["embedding"]

    target_faces = RetinaFace.extract_faces(img_path = img_path, align = True)
    target_img = target_faces[0]
    cv.imwrite(img_path + ".face.jpg", target_img[...,::-1])

    objs = DeepFace.analyze(
        img_path = img_path,
        actions = ['age', 'gender', 'race', 'emotion'],
        detector_backend='retinaface',
    )
    ID = uuid4().hex

    doc = {
        "id": ID,
        "title_vector": embedding,
        "title_name": fake.name(),
        'description': fake.text(),
        'img_path': img_path,
        'gender': objs[0].get('dominant_gender'),
        'dob': datetime.strptime(fake.date_of_birth(tzinfo=None, minimum_age=objs[0].get('age'), maximum_age=objs[0].get('age')).isoformat(), '%Y-%m-%d'),
        'birth_place': fake.city(),
        'image_url': 'http://localhost:8000/' + img_path,
        'face_path': img_path + ".face.jpg",
        'identified_age': objs[0].get('age'),
        'identified_gender': objs[0].get('dominant_gender'),
        'identified_race': objs[0].get('dominant_race'),
        'identified_emotion': objs[0].get('dominant_emotion'),
        'username': 'ferdous',
        'created_at': datetime.now(),
    }    
    es.create(index="suspects", id=ID, body=doc)

toc = time.time()
print("indexing completed in ", toc-tic, " seconds")

indexing public/police/2008026.jpg index 1 of 5910


Action: emotion: 100%|██████████| 4/4 [00:00<00:00, 16.72it/s]


indexing public/police/2101016.jpg index 2 of 5910


Action: emotion: 100%|██████████| 4/4 [00:00<00:00, 17.15it/s]


indexing public/police/2108032.jpg index 3 of 5910


...

Action: emotion: 100%|██████████| 4/4 [00:00<00:00, 16.76it/s]


indexing public/police/2006180.jpg index 158 of 5910


Action: emotion: 100%|██████████| 4/4 [00:02<00:00,  1.41it/s]


indexing public/police/2110090.jpg index 159 of 5910


KeyboardInterrupt: 

In [ ]:
objs = DeepFace.analyze(
            img_path = 'public/suspects2/2006023.jpg',
            actions = ['age', 'gender', 'race', 'emotion'],
            detector_backend='retinaface',
        )
objs

Action: emotion: 100%|██████████| 4/4 [00:00<00:00, 15.51it/s]


[{'age': 29,
  'region': {'x': 154,
   'y': 125,
   'w': 195,
   'h': 250,
   'left_eye': (300, 226),
   'right_eye': (206, 226),
   'nose': (253, 274),
   'mouth_left': (290, 315),
   'mouth_right': (214, 316)},
  'face_confidence': 1.0,
  'gender': {'Woman': np.float32(0.10216015), 'Man': np.float32(99.897835)},
  'dominant_gender': 'Man',
  'race': {'asian': np.float32(37.65101),
   'indian': np.float32(10.531604),
   'black': np.float32(6.94081),
   'white': np.float32(7.7297688),
   'middle eastern': np.float32(5.591658),
   'latino hispanic': np.float32(31.55515)},
  'dominant_race': 'asian',
  'emotion': {'angry': np.float32(1.1888607e-08),
   'disgust': np.float32(1.6202688e-05),
   'fear': np.float32(0.042208932),
   'happy': np.float32(91.80772),
   'sad': np.float32(4.452458e-05),
   'surprise': np.float32(1.1727671e-07),
   'neutral': np.float32(8.150021)},
  'dominant_emotion': 'happy'}]

In [12]:
objs = DeepFace.represent(img_path='public/suspects2/2006023.jpg', model_name='ArcFace', detector_backend='retinaface', align=True)
len(objs[0]["embedding"])

512